In [1]:
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras.losses import CategoricalCrossentropy
from network import NetCNN1D

import tqdm

from mne.decoding import CSP
from sklearn.model_selection import StratifiedKFold

import numpy as np
import sys, os

import dataset_BCICIV2a
from config_BCICIV2a import Config

import matplotlib.pyplot as plt


import pandas as pd


import pywt, cv2


from akida_models import fetch_file, akidanet_imagenet


config = Config()



# Choose from: 'CLeft', 'CRight', 'CUp' and 'CDown'
config.used_classes = ['CLeft', 'CRight']
session = 'S2'


config.t_start = 0 
config.t_end = 2.5

config.n_csp_components = 3




config.start = int(config.t_start * 250)
config.t_end = int(config.t_end * 250)

In [8]:
X_0, X_1 = [], []
for f in sorted(os.listdir('../EEG_RAW_DATA_NEW')):
    if config.used_classes[0] in f and session in f and "I01" in f:
        df = pd.read_csv(os.path.join('../EEG_RAW_DATA_NEW', f), delimiter=',')
        df = df.loc[:, ['C1', 'Cz', 'C2']]
        X_0.append(np.array(df)[config.t_start:config.t_end])
    if config.used_classes[1] in f and session in f and "I01" in f:
        df = pd.read_csv(os.path.join('../EEG_RAW_DATA_NEW', f), delimiter=',')
        df = df.loc[:, ['C1', 'Cz', 'C2']]
        X_1.append(np.array(df)[config.t_start:config.t_end])


X_0 = np.array(X_0).transpose(0, 2, 1)
X_1 = np.array(X_1).transpose(0, 2, 1)
print(np.array(X_0).shape)
print(np.array(X_1).shape)

y_0 = np.zeros(X_0.shape[0])
y_1 = np.ones(X_1.shape[0])

X, y = np.concatenate((X_0, X_1)), np.concatenate((y_0, y_1))

X = dataset_BCICIV2a.filter_rawEEG(X, config.lowcut, config.highcut)

(15, 3, 625)
(16, 3, 625)


In [9]:
def get_CWT(x):
    # x shape: (N, Channels, Time)

    time = np.linspace(0, 1, x.shape[2])
    widths = np.geomspace(1, 700, num=225)
    wavelet = "cmor1.5-1.0"
    sampling_period = np.diff(time).mean()
            
    cwtmatr, freqs = pywt.cwt(x, widths, wavelet, sampling_period=sampling_period)
    cwtmatr = np.abs(cwtmatr[:-1,:, :, :-1])
    

    x_cwt_resized = []
    for n in range(x.shape[0]):
        x_channels = []
        for ch in range(x.shape[1]):
            x_channels.append(cv2.resize(cwtmatr[:, n, ch], (224,224), interpolation=cv2.INTER_CUBIC))

        x_cwt_resized.append(np.array(x_channels))

    x_cwt_resized = np.array(x_cwt_resized)
    x_cwt_resized = (x_cwt_resized - np.mean(x_cwt_resized)) / np.std(x_cwt_resized)

    return x_cwt_resized




# Create a base model without top layers
base_model = akidanet_imagenet(input_shape=(224, 224, 3),
                               classes=2,
                               alpha=0.5,
                               include_top=False,
                               pooling='avg')

# Get pretrained quantized weights and load them into the base model
pretrained_weights = fetch_file(
    origin="https://data.brainchip.com/models/AkidaV2/akidanet/akidanet_imagenet_224_alpha_0.5.h5",
    fname="akidanet_imagenet_224_alpha_0.5.h5",
    cache_subdir='models')

base_model.load_weights(pretrained_weights, by_name=True)


from keras import Model
from keras.layers import Activation, Dropout, Reshape
from akida_models.layer_blocks import dense_block

x = base_model.output
x = dense_block(x,
                units=512,
                name='fc1',
                add_batchnorm=True,
                relu_activation='ReLU7.5')
x = Dropout(0.25, name='dropout_1')(x)
x = dense_block(x,
                units=2,
                name='predictions',
                add_batchnorm=False,
                relu_activation=False)
#x = Activation('softmax', name='act_softmax')(x)
#x = Reshape((2,), name='reshape')(x)

# Build the model
model_keras = Model(base_model.input, x, name='akidanet')



for layer in model_keras.layers:
    # Freeze all layers by default
    layer.trainable = False

    # Unfreeze 'fc1' and 'predictions' layers
    if layer.name == 'fc1' or layer.name == 'predictions' or layer.name == 'pw_separable_13' or layer.name == 'dw_separable_13':
        layer.trainable = True


model_keras.summary()



Model: "akidanet"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input (InputLayer)          [(None, 224, 224, 3)]     0         
                                                                 
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 conv_0 (Conv2D)             (None, 112, 112, 16)      432       
                                                                 
 conv_0/BN (BatchNormalizat  (None, 112, 112, 16)      64        
 ion)                                                            
                                                                 
 conv_0/relu (ReLU)          (None, 112, 112, 16)      0         
                                                                 
 conv_1 (Conv2D)             (None, 112, 112, 32)      4608      
                                                         

In [10]:

#X, y = dataset_BCICIV2a.subject_dataset(config, subject_id)
crossValidation_KF = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

acc = []

for train_index, valid_index in crossValidation_KF.split(X, y):
    # Reset model for each fold
    #model_keras = create_model()  # You need to define this function
    
    X_train = X[train_index]
    Y_train = y[train_index]
    X_valid = X[valid_index]
    Y_valid = y[valid_index]

    # Preprocess data once
    X_train_cwt = get_CWT(X_train)
    X_train_cwt = X_train_cwt.transpose(0, 3, 2, 1)
    X_valid_cwt = get_CWT(X_valid)
    X_valid_cwt = X_valid_cwt.transpose(0, 3, 2, 1)

    Y_train = tf.keras.utils.to_categorical(Y_train, num_classes=2)
    Y_valid = tf.keras.utils.to_categorical(Y_valid, num_classes=2)
    
    batch_size = 32
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train_cwt, Y_train))
    train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size)

    model_keras.compile(
        loss=CategoricalCrossentropy(from_logits=True, label_smoothing=0.0),
        optimizer=keras.optimizers.Adam(learning_rate=2e-4, weight_decay=1e-6), 
        metrics=[keras.metrics.CategoricalAccuracy(name='acc')]
    ) 

    # Use built-in training method
    history = model_keras.fit(
        train_dataset,
        validation_data=(X_valid_cwt, Y_valid),
        epochs=100,
        verbose=1
    )
    
    # Get final validation accuracy
    _, val_acc = model_keras.evaluate(X_valid_cwt, Y_valid, verbose=0)
    acc.append(val_acc)

Epoch 1/100

1/1 [==============================] - 4s 4s/step - loss: 0.8975 - acc: 0.5000 - val_loss: 0.8602 - val_acc: 0.4286
Epoch 2/100
1/1 [==============================] - 0s 234ms/step - loss: 0.9226 - acc: 0.4167 - val_loss: 0.8355 - val_acc: 0.5714
Epoch 3/100
1/1 [==============================] - 0s 236ms/step - loss: 0.9915 - acc: 0.2917 - val_loss: 0.8384 - val_acc: 0.5714
Epoch 4/100
1/1 [==============================] - 0s 227ms/step - loss: 0.9598 - acc: 0.3750 - val_loss: 0.8324 - val_acc: 0.5714
Epoch 5/100
1/1 [==============================] - 0s 223ms/step - loss: 1.0091 - acc: 0.4583 - val_loss: 0.8224 - val_acc: 0.5714
Epoch 6/100
1/1 [==============================] - 0s 228ms/step - loss: 0.9174 - acc: 0.5417 - val_loss: 0.8219 - val_acc: 0.5714
Epoch 7/100
1/1 [==============================] - 0s 231ms/step - loss: 0.9286 - acc: 0.4583 - val_loss: 0.8334 - val_acc: 0.4286
Epoch 8/100
1/1 [==============================] - 0s 230ms/step - loss: 0.8981 - acc